<a href="https://colab.research.google.com/github/LeanneMarieDP/CMSC124_LAB/blob/main/1DCNN_EncTrans_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1D CNN + Transformer Encoder

**Architecture:**
2. **1D CNN** — scans each frame's feature vector to recognize and learn local spatial patterns from landmarks, producing a compact per-frame summary
3. **Transformer Encoder** - reads ALL frame summaries at once (not sequentially) using **self-attention** to weight frames with high velocity/sudden changes and de-emphasize static frames

**Documentation**

- The 1D CNN acts as a **feature compressor**: it takes the raw 155-dim feature vector per frame and simplify and reduce it to essential features (joint angle spikes, velocity bursts, etc.)

- The Transformer Encoder uses **multi-head attention** to look at the entire sequence at once, so it never "forgets" early frames, critical for distinguishing pre-fall posture from the actual fall event

- **Together**, they capture both **local patterns** (CNN) and **global temporal dependencies** (Transformer)

**Training Parameters:**
- Learning Rate: 0.0001
- Optimizer: Adam
- Max Sequence Length: 64
- Batch Size: 16
- Epochs:60/70
- Patience: 7

- Dropout: 0.2

- Task: Binary classification (Fall vs No-Fall)


**Model Summary:**
 - Total parameters:     328,066
 - Trainable parameters: 328,066

In [ ]:
%pip install torch torchvision torchaudio scikit-learn matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from glob import glob
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import math
import warnings
warnings.filterwarnings('ignore')

 # check for GPU availability; CPU result
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration & Feature Selection

In [ ]:
# CONFIGURATION

DATASET_ROOT = './Split_Dataset_B_CSV'
# looks at 64 frames at a time (about 2 seconds)
MAX_SEQ_LEN = 64
BATCH_SIZE = 16
LEARNING_RATE = 0.00001
EPOCHS = 70
# for early stopping: if val loss doesn't improve for this many epochs, stop training
PATIENCE = 7

FALL_CLASSES = ['Front', 'Back', 'Left', 'Right']
NO_FALL_CLASSES = ['Sitting', 'Walking', 'Standing', 'Jumping']

# feature columns to extract from each CSV
# We select: normalized landmarks (99), velocity (24), acceleration (18),
# joint angles (8), trunk/neck inclination (2), trunk inclination velocity (1),
# bbox width & height (2) = 154 features per frame

LANDMARK_NAMES = [
    'NOSE', 'LEFT_EYE_INNER', 'LEFT_EYE', 'LEFT_EYE_OUTER',
    'RIGHT_EYE_INNER', 'RIGHT_EYE', 'RIGHT_EYE_OUTER',
    'LEFT_EAR', 'RIGHT_EAR', 'MOUTH_LEFT', 'MOUTH_RIGHT',
    'LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_ELBOW', 'RIGHT_ELBOW',
    'LEFT_WRIST', 'RIGHT_WRIST', 'LEFT_PINKY', 'RIGHT_PINKY',
    'LEFT_INDEX', 'RIGHT_INDEX', 'LEFT_THUMB', 'RIGHT_THUMB',
    'LEFT_HIP', 'RIGHT_HIP', 'LEFT_KNEE', 'RIGHT_KNEE',
    'LEFT_ANKLE', 'RIGHT_ANKLE', 'LEFT_HEEL', 'RIGHT_HEEL',
    'LEFT_FOOT_INDEX', 'RIGHT_FOOT_INDEX'
]

# used in velocity/acceleration calculations based in angle/position changes
KEY_LANDMARKS = ['NOSE', 'LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_HIP', 'RIGHT_HIP',
'LEFT_KNEE', 'RIGHT_KNEE', 'LEFT_ANKLE', 'RIGHT_ANKLE']

# centralize the definition of which features we are using, so it's easier to maintain and update
def get_feature_columns():
    # Build the list of feature columns to extract from each CSV.
    cols = []

    # Normalized landmark coordinates (33 landmarks x 3 axes = 99)
    for name in LANDMARK_NAMES:
        cols.extend([f'{name}_norm_x', f'{name}_norm_y', f'{name}_norm_z'])

    # Joint angles (8)
    cols.extend([
        'left_elbow_angle', 'right_elbow_angle',
        'left_shoulder_angle', 'right_shoulder_angle',
        'left_hip_angle', 'right_hip_angle',
        'left_knee_angle', 'right_knee_angle',
    ])

    # Trunk and neck inclination (2)
    cols.extend(['trunk_inclination', 'neck_inclination'])

    # Velocity features for key landmarks (9 landmarks x 2 axes = 18)
    for name in KEY_LANDMARKS:
        cols.extend([f'{name}_vel_x', f'{name}_vel_y'])

    # Hip center and bbox center velocity (6)
    cols.extend([
        'hip_center_vel_x', 'hip_center_vel_y', 'hip_center_vel_magnitude',
        'bbox_center_vel_x', 'bbox_center_vel_y', 'bbox_center_vel_magnitude',
    ])

    # Acceleration features for key landmarks (9 landmarks x 2 axes = 18)
    for name in KEY_LANDMARKS:
        cols.extend([f'{name}_acc_x', f'{name}_acc_y'])

    # Trunk inclination velocity (1)
    cols.append('trunk_inclination_vel')

    # Bbox dimensions (2)
    cols.extend(['bbox_width', 'bbox_height'])

    return cols


FEATURE_COLUMNS = get_feature_columns()
NUM_FEATURES = len(FEATURE_COLUMNS)
print(f"Number of input features per frame: {NUM_FEATURES}")
print(f"Max sequence length (frames): {MAX_SEQ_LEN}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Epochs: {EPOCHS} (early stopping patience={PATIENCE})")

## Load CSVs and Create PyTorch DataLoader

Each CSV is one sequence (one video clip):
1. Load all CSVs from Train/Test/Validation splits
2. Extract only the selected feature columns
3. Label: `1` = Fall, `0` = No-Fall (binary)
4. Pad short sequences with zeros / truncate long sequences to `MAX_SEQ_LEN`
5. Create a binary mask so the model ignores padded frames

In [ ]:
class FallDetectionDataset(Dataset):
    # PyTorch Dataset that loads pre-extracted CSV sequences for fall detection.

    def __init__(self, data_root, split, feature_cols, max_seq_len, fall_classes):
        """
        Args:
            data_root: Path to Split_Dataset_A_CSV/
            split: 'Training', 'Testing', or 'Validation'
            feature_cols: List of column names to use as model features
            max_seq_len: Pad/truncate sequences to this length
            fall_classes: List of activity names that count as 'Fall'
        """

        self.feature_cols = feature_cols
        self.max_seq_len = max_seq_len
        self.samples = []  # list of (csv_path, label)

        # check if split directory exists
        split_dir = os.path.join(data_root, split)
        if not os.path.exists(split_dir):
            print(f"WARNING: Split directory not found: {split_dir}")
            return

        # Iterate through activities and CSV files, assign labels based on fall_classes
        for activity in sorted(os.listdir(split_dir)):
            activity_dir = os.path.join(split_dir, activity)
            if not os.path.isdir(activity_dir):
                continue
            label = 1 if activity in fall_classes else 0
            for csv_file in sorted(glob(os.path.join(activity_dir, '*.csv'))):
                self.samples.append((csv_file, label))

        # Count class distribution
        fall_count = sum(1 for _, l in self.samples if l == 1)
        nofall_count = len(self.samples) - fall_count
        print(f"  [{split}] Loaded {len(self.samples)} sequences "
              f"(Fall={fall_count}, No-Fall={nofall_count})")

    def __len__(self):
        return len(self.samples)

    # load a sequence, extract features, pad/truncate to max_seq_len, and return (features, mask, label)
    def __getitem__(self, idx):
        csv_path, label = self.samples[idx]
        df = pd.read_csv(csv_path)

        # extract only the feature columns that exist in this CSV
        available_cols = [c for c in self.feature_cols if c in df.columns]
        missing_cols = [c for c in self.feature_cols if c not in df.columns]

        features = df[available_cols].values.astype(np.float32)

        # if some columns are missing (undetected CSV), add zero columns
        # fill with zero the missing columns
        if missing_cols:
            zeros = np.zeros((len(df), len(missing_cols)), dtype=np.float32)
            features = np.hstack([features, zeros])

        # Replace any remaining NaN with 0
        features = np.nan_to_num(features, nan=0.0)

        seq_len = features.shape[0]

        # Pad or truncate to max_seq_len
        # mask: 1 for real frames, 0 for padded frames (for attention masking in transformer)
        if seq_len >= self.max_seq_len:
            features = features[:self.max_seq_len]
            mask = np.ones(self.max_seq_len, dtype=np.float32)
        else:
            pad_len = self.max_seq_len - seq_len
            features = np.pad(features, ((0, pad_len), (0, 0)), mode='constant')
            mask = np.concatenate([np.ones(seq_len, dtype=np.float32),np.zeros(pad_len, dtype=np.float32)])

        return (torch.tensor(features),          # (max_seq_len, num_features)
                torch.tensor(mask),              # (max_seq_len,)
                torch.tensor(label, dtype=torch.long))

# create datasets
print("LOADING DATASETS")
train_dataset = FallDetectionDataset(DATASET_ROOT, 'Training', FEATURE_COLUMNS, MAX_SEQ_LEN, FALL_CLASSES)
val_dataset = FallDetectionDataset(DATASET_ROOT, 'Validation', FEATURE_COLUMNS, MAX_SEQ_LEN, FALL_CLASSES)
test_dataset = FallDetectionDataset(DATASET_ROOT, 'Testing', FEATURE_COLUMNS, MAX_SEQ_LEN, FALL_CLASSES)

# create dataloaders
# to create batches and shuffle the training data
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# display dataset and dataloader summary
print(f"\nDataLoader summary:")
print(f"  Train: {len(train_dataset)} sequences, {len(train_loader)} batches")
print(f"  Val:   {len(val_dataset)} sequences, {len(val_loader)} batches")
print(f"  Test:  {len(test_dataset)} sequences, {len(test_loader)} batches")

# check a sample batch from the training dataloader
x, mask, y = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  Features: {x.shape}  (batch, seq_len, features)")
print(f"  Mask:     {mask.shape}")
print(f"  Labels:   {y.shape} -> {y.tolist()}")

## Model Architecture — 1D CNN + Transformer Encoder

**Data flow:**
```
Input: (batch, 64 frames, 154 features)
        │
  ┌─────▼─────┐
  │  1D CNN    │  ← Conv1D layers extract local patterns per frame
  │  Block     │    (sudden angle changes, velocity spikes)
  └─────┬─────┘
        │  Output: (batch, 64, d_model)
  ┌─────▼──────────┐
  │  Positional     │  ← Adds frame-order information since
  │  Encoding       │    Transformer has no inherent sequence sense
  └─────┬──────────┘
  ┌─────▼──────────┐
  │  Transformer    │  ← Multi-head self-attention reads ALL frames
  │  Encoder        │    at once, weighting high-velocity frames
  └─────┬──────────┘
        │  Output: (batch, 64, d_model)
  ┌─────▼─────┐
  │  Global    │  ← Average over non-padded frames
  │  Avg Pool  │
  └─────┬─────┘
        │  Output: (batch, d_model)
  ┌─────▼─────┐
  │  FC Head   │  ← Binary classification: Fall / No-Fall
  └─────┬─────┘
        │
    Output: (batch, 2)
```

In [ ]:
class PositionalEncoding(nn.Module):
    #  positional encoding so the Transformer knows frame order.
    # inject informaational about the position of each frame in the sequence.


    ## initialize the positional encoding matrix based on sine and cosine functions of different frequencies.
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # unsqueeze to add batch dimension, so pe can be added directly to input embeddings
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    # add positional encoding to input embeddings
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1), :]

class FallDetectionCNNTransformer(nn.Module):
    """
    1D CNN + Transformer Encoder for fall detection.

    1D CNN: Extracts local patterns from the raw feature sequence.
    Transformer Encoder: Applies multi-head self-attention to capture
        global temporal dependencies across all frames simultaneously.
    """

    # initialize the model layers: 1D CNN for local feature extraction, positional encoding, transformer encoder layers, and a classification head.
    def __init__(self, num_features, d_model=128, nhead=4, num_encoder_layers=2,
                 dim_feedforward=256, dropout=0.1, num_classes=2):
        super().__init__()

        #  1D CNN Block
        # Input: (batch, seq_len, num_features) -> transpose to (batch, num_features, seq_len)

        # Conv1D operates:
        # it applies convolutional filters across the sequence of frames for each feature, allowing it to learn local temporal patterns (like sudden changes in velocity or specific joint angle configurations) that are indicative of falls.

        self.cnn = nn.Sequential(
            # Layer 1: Capture immediate frame-level patterns
            nn.Conv1d(num_features, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            # Layer 2: Capture slightly wider temporal patterns (3-frame window)
            nn.Conv1d(64, d_model, kernel_size=3, padding=1),
            nn.BatchNorm1d(d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )


        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len=MAX_SEQ_LEN)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            # number of attention heads in the multi-head attention mechanism.
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # (batch, seq, feature) format
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_encoder_layers
        )

        # Classification
        # stack the layers together in a sequential block for the final classification
        self.classifier = nn.Sequential(
            # reduce dimensionality to 64 for the final classification layers
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            # after the CNN and transformer layers have processed the sequence,
            # it will squash all information into a single vector: 0 for No-Fall, 1 for Fall
            nn.Linear(64, num_classes)
        )


    # define the forward pass through the model
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, num_features) — raw feature sequence
            mask: (batch, seq_len) — 1.0 for real frames, 0.0 for padding
        Returns:
            logits: (batch, num_classes)
        """
        # 1D CNN
        # Transpose: (batch, seq_len, features) -> (batch, features, seq_len)
        x = x.transpose(1, 2)
        x = self.cnn(x)
        # Transpose back: (batch, d_model, seq_len) -> (batch, seq_len, d_model)
        x = x.transpose(1, 2)

        # Positional Encoding
        x = self.pos_encoder(x)

        # Transformer Encoder

        # The mask is used to ignore padded elements in the sequence during attention computation.

        # Create key_padding_mask: True = ignore, False = attend
        if mask is not None:
            key_padding_mask = (mask == 0)  # True where padded
        else:
            key_padding_mask = None

        x = self.transformer_encoder(x, src_key_padding_mask=key_padding_mask)

        # Masked Global Average Pooling
        # Average only over non-padded frames
        if mask is not None:
            mask_expanded = mask.unsqueeze(-1)  # (batch, seq_len, 1)
            x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        else:
            x = x.mean(dim=1)

        # Classification
        logits = self.classifier(x)
        return logits


# Instantiate model
model = FallDetectionCNNTransformer(
    num_features=NUM_FEATURES,
    d_model=128,
    nhead=4,
    num_encoder_layers=2,
    dim_feedforward=256,
    dropout=0.2,
    num_classes=2
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Summary:")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"\n{model}")



## Training Loop with Early Stopping

In [ ]:
# Loss, Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

# Early stopping state
best_val_loss = float('inf')
best_model_state = None
patience_counter = 0

print("Starting training...")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9} | {'Status'}")
print("-" * 75)

for epoch in range(1, EPOCHS + 1):
    # Training

    """In each epoch, the model goes through the entire training dataset in batches.
    For each batch, it performs a forward pass to compute the predictions,
    calculates the loss using the criterion,
    and then performs a backward pass to compute gradients.
    The optimizer then updates the model parameters based on these gradients.
    After processing all batches, it computes the average training loss and accuracy for that epoch.
    """

    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for features, mask, labels in train_loader:
        features = features.to(device)
        mask = mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(features, mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for features, mask, labels in val_loader:
            features = features.to(device)
            mask = mask.to(device)
            labels = labels.to(device)

            logits = model(features, mask)
            loss = criterion(logits, labels)

            val_loss_sum += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        status = "improved!"
    else:
        patience_counter += 1
        status = f"  patience {patience_counter}/{PATIENCE}"

    print(f"{epoch:>5} | {train_loss:>10.4f} | {train_acc:>8.1%} | {val_loss:>10.4f} | {val_acc:>8.1%} | {status}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nRestored best model (val_loss={best_val_loss:.4f})")

# Save model
MODEL_SAVE_PATH = 'fall_detection_cnn_transformer.pth'
torch.save({
    'model_state_dict': best_model_state,
    'feature_columns': FEATURE_COLUMNS,
    'num_features': NUM_FEATURES,
    'max_seq_len': MAX_SEQ_LEN,
    'model_config': {
        'd_model': 128, 'nhead': 4, 'num_encoder_layers': 2,
        'dim_feedforward': 256, 'dropout': 0.2, 'num_classes': 2
    }
}, MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# loss
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# accuracy
axes[1].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest validation loss: {best_val_loss:.4f}")
print(f"Best validation accuracy: {max(history['val_acc']):.1%}")

## Evaluation on Test Set

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for features, mask, labels in test_loader:
        features = features.to(device)
        mask = mask.to(device)

        logits = model(features, mask)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# evaluate and print results
print("TEST SET EVALUATION\n\n")
print(f"\nAccuracy: {accuracy_score(all_labels, all_preds):.1%}\n")
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['No-Fall', 'Fall']))

# Create confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['No-Fall', 'Fall'], yticklabels=['No-Fall', 'Fall'],
       xlabel='Predicted', ylabel='Actual',
       title='Confusion Matrix — Test Set')

# annotate cells
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=16)

plt.tight_layout()
plt.show()

## Per-Activity Breakdown

Check how the model performs on each individual activity (not just Fall vs No-Fall).

In [ ]:
# er-activity accuracy on the test set
model.eval()
activity_results = {}

test_split_dir = os.path.join(DATASET_ROOT, 'Testing')
for activity in sorted(os.listdir(test_split_dir)):
    activity_dir = os.path.join(test_split_dir, activity)
    if not os.path.isdir(activity_dir):
        continue

    label = 1 if activity in FALL_CLASSES else 0
    preds_list = []

    for csv_file in sorted(glob(os.path.join(activity_dir, '*.csv'))):
        df = pd.read_csv(csv_file)
        available_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
        missing_cols = [c for c in FEATURE_COLUMNS if c not in df.columns]

        features = df[available_cols].values.astype(np.float32)
        if missing_cols:
            features = np.hstack([features, np.zeros((len(df), len(missing_cols)), dtype=np.float32)])
        features = np.nan_to_num(features, nan=0.0)

        seq_len = features.shape[0]
        if seq_len >= MAX_SEQ_LEN:
            features = features[:MAX_SEQ_LEN]
            mask = np.ones(MAX_SEQ_LEN, dtype=np.float32)
        else:
            pad_len = MAX_SEQ_LEN - seq_len
            features = np.pad(features, ((0, pad_len), (0, 0)), mode='constant')
            mask = np.concatenate([np.ones(seq_len, dtype=np.float32),np.zeros(pad_len, dtype=np.float32)])

        x = torch.tensor(features).unsqueeze(0).to(device)
        m = torch.tensor(mask).unsqueeze(0).to(device)

        with torch.no_grad():
            logit = model(x, m)
            pred = logit.argmax(dim=1).item()
            preds_list.append(pred)

    correct = sum(1 for p in preds_list if p == label)
    total = len(preds_list)
    acc = correct / total if total > 0 else 0
    activity_results[activity] = {'correct': correct, 'total': total, 'accuracy': acc, 'label': label}

print(f"{'Activity':<12} {'Type':<8} {'Correct':>7} / {'Total':>5} {'Accuracy':>10}")
print("-" * 50)
for activity in FALL_CLASSES + NO_FALL_CLASSES:
    if activity in activity_results:
        r = activity_results[activity]
        lbl = 'Fall' if r['label'] == 1 else 'No-Fall'
        print(f"{activity:<12} {lbl:<8} {r['correct']:>7} / {r['total']:>5} {r['accuracy']:>9.1%}")